In [ ]:
!pip install torch==2.4.0+cu118 torchvision==0.19.0+cu118 torchaudio==2.4.0+cu118 --index-url https://download.pytorch.org/whl/cu118


In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

In [ ]:
import os


os.listdir('/kaggle/input/datasets/sarkisshilgevorkyan/coco-dataset-for-yolo')

In [ ]:
img_dir = "/kaggle/input/datasets/sarkisshilgevorkyan/coco-dataset-for-yolo/coco/images/test2017"
print(len(os.listdir(img_dir)))

In [ ]:
print(os.listdir('/kaggle/input/datasets/sarkisshilgevorkyan/coco-dataset-for-yolo/coco/images/test2017')[:5])

In [ ]:


label_dir = "/kaggle/input/datasets/sarkisshilgevorkyan/coco-dataset-for-yolo/coco/labels/train2017"

empty = 0
total = 0

for file in os.listdir(label_dir):
    total += 1
    if os.path.getsize(os.path.join(label_dir, file)) == 0:
        empty += 1

print("Total labels:", total)
print("Empty labels:", empty)

In [ ]:
import json

with open("/kaggle/input/datasets/sarkisshilgevorkyan/coco-dataset-for-yolo/coco/annotations/instances_train2017.json") as f:
    data = json.load(f)

print(len(data["categories"]))

In [ ]:
import os

base = "/kaggle/input/datasets/sarkisshilgevorkyan/coco-dataset-for-yolo/coco"
print(os.listdir(base))

In [ ]:
print(os.listdir(base + "/images"))
print(os.listdir(base + "/labels"))

In [ ]:
data_yaml = """
path: /kaggle/input/datasets/sarkisshilgevorkyan/coco-dataset-for-yolo/coco

train: images/train2017
val: images/val2017

nc: 80

names: [
'person','bicycle','car','motorcycle','airplane','bus','train','truck','boat',
'traffic light','fire hydrant','stop sign','parking meter','bench','bird','cat',
'dog','horse','sheep','cow','elephant','bear','zebra','giraffe','backpack',
'umbrella','handbag','tie','suitcase','frisbee','skis','snowboard','sports ball',
'kite','baseball bat','baseball glove','skateboard','surfboard','tennis racket',
'bottle','wine glass','cup','fork','knife','spoon','bowl','banana','apple',
'sandwich','orange','broccoli','carrot','hot dog','pizza','donut','cake','chair',
'couch','potted plant','bed','dining table','toilet','tv','laptop','mouse',
'remote','keyboard','cell phone','microwave','oven','toaster','sink',
'refrigerator','book','clock','vase','scissors','teddy bear','hair drier','toothbrush'
]
"""

with open('/kaggle/working/data.yaml', 'w') as f:
    f.write(data_yaml)

In [ ]:
model_8 = YOLO("yolov8n-seg.pt")  # runs perfectly on GPU

In [ ]:
model_26=YOLO("yolo26n-seg.pt")

In [ ]:


import random
import shutil
import matplotlib.pyplot as plt
import cv2

# --- 1. SETUP & TRAINING ---
# Note: Using yolo11 as the current latest stable version from Ultralytics
model_8 = YOLO("yolov8n-seg.pt")
model_26 = YOLO("yolo26n-seg.pt")  

# Train YOLOv8
model_8.train(
    data="/kaggle/working/data.yaml",
    epochs=10,
    imgsz=640,
    name="v8_run"
)

# Train YOLO26
model_26.train(
    data="/kaggle/working/data.yaml",
    epochs=10,
    imgsz=640,
    name="v26_run"
)

# --- 2. ORGANIZE WEIGHTS ---
shutil.copy("/kaggle/working/runs/segment/v8_run/weights/best.pt", "/kaggle/working/best_8.pt")
shutil.copy("/kaggle/working/runs/segment/v26_run/weights/best.pt", "/kaggle/working/best_26.pt")

# Load the best versions
model_8_final = YOLO("/kaggle/working/best_8.pt")
model_26_final = YOLO("/kaggle/working/best_26.pt")

# --- 3. RANDOM TEST SELECTION ---
test_path = "/kaggle/input/datasets/sarkisshilgevorkyan/coco-dataset-for-yolo/coco/images/test2017"
all_images = [os.path.join(test_path, f) for f in os.listdir(test_path) if f.endswith(('.jpg', '.png'))]
random_10 = random.sample(all_images, 10)

# --- 4. PREDICT & DISPLAY ---
print("\n--- Displaying Results for 10 Random Images ---")
for img_path in random_10:
    res8 = model_8_final.predict(source=img_path, conf=0.25, save=False)[0]
    res26 = model_26_final.predict(source=img_path, conf=0.25, save=False)[0]
    
    # Create side-by-side plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
    
    ax1.imshow(cv2.cvtColor(res8.plot(), cv2.COLOR_BGR2RGB))
    ax1.set_title(f"YOLOv8: {os.path.basename(img_path)}")
    ax1.axis('off')
    
    ax2.imshow(cv2.cvtColor(res26.plot(), cv2.COLOR_BGR2RGB))
    ax2.set_title(f"YOLO26: {os.path.basename(img_path)}")
    ax2.axis('off')
    
    plt.show()


In [ ]:
metrics = model_26_final.val()
output_path = "val_results_8.txt"
with open(output_path, "w") as f:
    f.write(f"Precision: {metrics.box.mp}\n")
    f.write(f"Recall: {metrics.box.mr}\n")
    f.write(f"mAP50: {metrics.box.map50}\n")
    f.write(f"mAP50-95: {metrics.box.map}\n")

print(f"Results saved to kaggle/output/{output_path}")